# 接触可能性探索の評価
## このnotebookで行う事
+ 接触可能性探索のパラメータを変えたときの死角をチェックする

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

## プロジェクトのルートに移動
+ ルートに移動することで、python -m argus_synchroの実行環境と同じように実行が可能

In [ ]:
import os
os.chdir("../")

## 必要なライブラリの読み込み

In [ ]:
import numpy as np
from numpy.typing import NDArray

from octotree.collision_detector import LayerBasedCollisionDetector, CoordMethod

import argus_synchro.SubScrutinizer as SubScrt
from argus_synchro.interface.octotree_func import OctoTreeFuncOn

## AppConfigの読み込み
+ 書き換えたいパラメータは書き換えられるように関数を読み込んでおく

In [ ]:
from configparser import ConfigParser, ExtendedInterpolation
from argus_synchro.experiments.config_replace import with_frozen_app_config
from argus_synchro.config.app_config import AppConfig

In [ ]:
app_ini = ConfigParser(interpolation=ExtendedInterpolation())
app_ini.read("./config/settings.ini", "UTF-8")
# 共有メモリに反映
app_config = AppConfig(app_ini)

## その他処理に必要なインスタンスを生成
+ 八分木にデータを入れたりするインスタンスと衝突判定インスタンスが必要なので、生成する

In [ ]:
octotree_func = OctoTreeFuncOn()

In [ ]:
coord_method = CoordMethod.from_string(app_config.CollisionDetection.coord_method)
collision_detector = LayerBasedCollisionDetector(coord_method=coord_method)

# 接触可能性探索に必要なデータの読み込み

## 最短部位計算に用いる機体点群を取得する

### 読み込むファイルのチェック
+ 意図しない機体点群を呼んでないか出力しておく

In [ ]:
(
    app_config.OctoTree.col_machine_dir, 
    app_config.LiDARPosition,
    app_config.OctoTree.json_col_machine_file,
)

In [ ]:
_, machine_mobile_points_measure, machine_immobile_points_measure = SubScrt.create_machine_points(
    machine_dir=app_config.OctoTree.col_machine_dir, 
    lidarposition=app_config.LiDARPosition,
    json_file=app_config.OctoTree.json_col_machine_file,
)

## 最短部位計算の機体点群を可視化
+ これも念のため実施

In [ ]:
import k3d
from argus_synchro.experiments.debug_vis.viewer_3d import create_simple_k3d_points

In [ ]:
plot = k3d.plot()

plot += create_simple_k3d_points(machine_immobile_points_measure, point_size=0.02)
plot += create_simple_k3d_points(machine_mobile_points_measure, point_size=0.02)

plot.display()

## 変えたいパラメータの設定
+ とりあえず、旋回角のルックアップテーブルの数だけ色々変えられるように作るが、他のパラメータも振れるようにしておく

In [ ]:
from itertools import product

In [ ]:
def dict_combinations(data):
    """
    辞書の中の各valueのリストの組み合わせを生成する
    """
    # Validate input
    if not isinstance(data, dict):
        raise TypeError("Input must be a dictionary.")
    for k, v in data.items():
        if not isinstance(v, (list, tuple)):
            raise TypeError(f"Value for key '{k}' must be a list or tuple.")
        if not v:  # Empty list means no combinations possible
            return []

    # Extract keys and value lists
    keys = list(data.keys())
    values_lists = [data[k] for k in keys]

    # Generate Cartesian product
    combinations = []
    for combo in product(*values_lists):
        yield dict(zip(keys, combo))

## 各種振るパラメータと、それに対するふり幅をここで定義

In [ ]:
params = {
    "key_num": [4, 8, 16, 32, 64],
}

# パラメータを振りながら死角の有無を評価する
## 評価方法
+ 評価用の旋回角だけ機体が旋回した時のその周囲に点群を生成して、生成した点群を検出できるか評価する

In [ ]:
from collections import ChainMap

import pandas as pd

from octotree import controller as octo_ctrl
from octotree.octotree import NodeEntity, NodeClusterKey

from argus_synchro.py_octotree.detectable_points import get_detectable_z_range, create_eval_data_cylinder, _circle_modulo, DetectableCylinderPointBase
from argus_synchro.common.common import rotate_machine

In [ ]:
def eval_detectable(
    eval_points: NDArray,
    yaw_angle: float,
    machine_immobile_points_measure: NDArray,
    machine_mobile_points_measure: NDArray,
    det_point_immobile_gen: DetectableCylinderPointBase,
    det_point_mobile_gen: DetectableCylinderPointBase,
    collision_detector: LayerBasedCollisionDetector,
    window: int,
    app_config: AppConfig,
) -> tuple[NDArray, NDArray]:
    """ eval_pointsに対する接触可能性探索の評価
    接触可能性探索が失敗した部分と成功した部分のLiDAR座標の点群が返ってくる
    """
    # 八分木の初期化
    octree_obj = SubScrt.initialize_octotree(
        machine_immobile_points_measure=machine_immobile_points_measure,
        machine_mobile_points_measure=machine_mobile_points_measure,
        max_xyz=app_config.OctoTree.max_xyz,
        min_xyz=app_config.OctoTree.min_xyz,
        max_tree_depth=app_config.OctoTree.max_tree_depth,
        use_node_stats=app_config.OctoTree.use_node_stats,
        dialate_point_size=app_config.CollisionDetection.dialate_point_size,
        origin_w2oct=(0.0, 0.0, 0.0)
    )

    # 機体点群の非可動部を八分木に入れる
    octree_obj = SubScrt.put_immobile_points_to_octotree(
        octotree_obj=octree_obj,
        machine_immobile_points_measure=machine_immobile_points_measure,
        machine_immobile_points_detect=det_point_immobile_gen.get_detectable_points(yaw_angle=None),
        machine_center=app_config.machine.offset_rotate_center,
    )

    # 評価用点群を八分木に入れる
    downsampled_eval_points, octree_obj = octotree_func.octotree_accum(
        accum_points=eval_points,
        octotree_obj_pcd=octree_obj,
        target_entity=NodeEntity.OTHER,
        point_depth=app_config.OctoTree.clustering_tree_depth,
    )

    # 機体点群の可動部を八分木に入れる
    octree_obj = octotree_func.update_machine_mobile(
        machine_mobile_points_measure=machine_mobile_points_measure,
        machine_mobile_points_detect=det_point_mobile_gen.get_detectable_points(yaw_angle=yaw_angle),
        octotree_obj=octree_obj,
        yaw_angle=yaw_angle,
    )

    # 接触可能性探索に関連するOctoMap: dict[離散座標, 八分木ノード]を取り出す
    ## このkeyが接触可能性探索に該当する
    machine_detect_key = [
        NodeClusterKey(entity=NodeEntity.CRANE_IMMOBILE_FOR_DET, cluster_id=None),
        NodeClusterKey(entity=NodeEntity.CRANE_MOBILE_FOR_DET, cluster_id=None),
    ]
    machine_octnodes = dict(
        ChainMap(
            *[
                octree_obj.entity_octonodes[key]
                for key in machine_detect_key
            ]
        )
    )
    
    # 機体点群をwindowだけ上位に上る
    src_coords = collision_detector.create_dialation_coord(machine_octnodes, window)
    
    # LiDAR点群のOctoMapを取り出す
    pcd_octonodes = octree_obj.entity_octonodes[NodeClusterKey(NodeEntity.OTHER, None)]
    
    # 評価点群の各点が機体点群と被っているか判定する
    pcd_isin_machine = {
        vox_coord: oct_nodes.morton_code >> (window * 3) in src_coords
        for vox_coord, oct_nodes in pcd_octonodes.items()
    }
    
    # 機体と被っている点の離散座標を取得する
    vox_coords_fail = np.array(
        list(dict(filter(lambda x: not x[1], pcd_isin_machine.items())).keys())
    )

    # 機体と被っていない点の離散座標を取得する
    vox_coords_success = np.array(
        list(dict(filter(lambda x: x[1], pcd_isin_machine.items())).keys())
    )
    
    # 離散座標を八分木座標に変換する
    oct_coords_fail = octree_obj.vox2oct_coords(vox_coords_fail)
    oct_coords_success = octree_obj.vox2oct_coords(vox_coords_success)
    
    return oct_coords_fail, oct_coords_success

## 評価データの設定値

In [ ]:
n_test_angles = 360

## 各パラメータで接触可能性探索の評価を行う
### 概要
1. 振りたいパラメータでCollisionDetectionを更新
2. ループ内で何度も呼んでいるパラメータを変数に設定する
3. そのパラメータで接触可能性探索の点群を生成するインスタンスを生成する
4. 評価対象のyaw_angleに対して、以下のA,B,Cの処理を繰り返す
    1. 評価対象のyaw_angleだけ機体が旋回している想定で、その周辺に評価用のLiDAR点群を生成
    2. その評価用の各LiDAR点群に対して接触可能性探索を行って、接触可能性有無で点群を分ける
    3. 分けたそれぞれをtupleにしてlistに格納
5. まとめた結果をDataFrameに格納して次のパラメータのペアに移って、再度1の処理を行う

In [ ]:
df_angle_eval = pd.DataFrame(
    columns=["yaw_angle", "eval_method", "n_fail", "n_success", "key_num"]
)
for param_pair in dict_combinations(params):
    # 1. 振りたいパラメータでCollisionDetectionを更新
    target_conf = with_frozen_app_config(
        conf=app_config.CollisionDetection,
        **param_pair
    )
    print(f"target_conf = {target_conf}")
    app_config.CollisionDetection = target_conf

    # 2. ループ内で何度も呼んでいるパラメータを変数に設定する
    z_range=get_detectable_z_range(app_config.General, app_config.CollisionDetection)
    max_dist=app_config.CollisionDetection.max_dist
    grid_intervals=app_config.CollisionDetection.grid_intervals
    min_radius=app_config.CollisionDetection.min_radius
    max_radius=app_config.CollisionDetection.max_radius
    vis_tree_depth = app_config.OctoTree.max_tree_depth - app_config.CollisionDetection.dialate_point_size
    window = app_config.CollisionDetection.dialate_point_size
    key_num = app_config.CollisionDetection.key_num

    # 3. そのパラメータで接触可能性探索の点群を生成するインスタンスを生成する
    det_point_mobile_gen, det_point_immobile_gen = SubScrt.initialize_detectable_point_generators(
        machine_mobile_points=machine_mobile_points_measure,
        machine_immobile_points=machine_immobile_points_measure,
        detectable_tree_depth=vis_tree_depth,
        z_range=z_range,
        max_dist=max_dist,
        grid_intervals=grid_intervals,
        min_radius=min_radius,
        max_radius=max_radius,
        key_num=app_config.CollisionDetection.key_num,
        octotree_conf=app_config.OctoTree,
        dialate_point_size=app_config.CollisionDetection.dialate_point_size,
        offset_rotate_center=app_config.machine.offset_rotate_center,
    )

    # 4. 評価対象のyaw_angleに対して、以下のA,B,Cの処理を繰り返す
    test_yaw_angles = np.unique(_circle_modulo(np.linspace(0, 2 * np.pi, num=n_test_angles, endpoint=False)))
    res_angles = []
    for i, test_yaw_angle in enumerate(test_yaw_angles):
        # A. 評価対象のyaw_angleだけ機体が旋回している想定で、その周辺に評価用のLiDAR点群を生成
        _test_yaw_angle = _circle_modulo(test_yaw_angle)
        # test_yaw_angleに応じて、機体点群は回るので、それに応じた機体点群から距離max_distに位置する点群を作る
        eval_points = create_eval_data_cylinder(
            machine_points=np.vstack([
                rotate_machine(machine_mobile_points_measure, -_test_yaw_angle, np.array([0.0, 0.0, 0.0])),
                machine_immobile_points_measure,
            ]),
            z_range=z_range,
            grid_intervals=grid_intervals,
            min_radius=min_radius,
            max_radius=max_radius,
            max_dist=max_dist,
        )

        # B. その評価用の各LiDAR点群に対して接触可能性探索を行って、接触可能性有無で点群を分ける
        oct_coords_fail, oct_coords_success = eval_detectable(
            eval_points=eval_points,
            yaw_angle=_test_yaw_angle,
            machine_immobile_points_measure=machine_immobile_points_measure,
            machine_mobile_points_measure=machine_mobile_points_measure,
            det_point_immobile_gen=det_point_immobile_gen,
            det_point_mobile_gen=det_point_mobile_gen,
            collision_detector=collision_detector,
            window=window,
            app_config=app_config,
        )
        
        # C. 分けたそれぞれをtupleにしてlistに格納
        res_angles.append(
            (_test_yaw_angle, "rot_table", len(oct_coords_fail), len(oct_coords_success), key_num)
        )

    #5. まとめた結果をDataFrameに格納して次のパラメータのペアに移って、再度1の処理を行う
    df_angle_eval = pd.concat(
        [
            pd.DataFrame(res_angles, columns=df_angle_eval.columns),
            df_angle_eval,
        ],
        ignore_index=True,
    )
    print(f"key_num = {key_num} is finished.")
    print("------- next params ---------")

## 結果を集約
+ DataFrameはyaw_angle, eval_method, n_fail, n_success, key_numを列に持っていて、key_num毎の失敗数や成功数を集約すれば大まかな結果は分かるので、それで確認

In [ ]:
df_angle_eval.groupby("key_num")[["n_fail", "n_success"]].agg(["sum", "mean"])

# 原因を調べる
+ 特定のyaw_angle, key_numで弱い事を確認して必要な対策を考える

## 上手くいっていないkey_numを確認

In [ ]:
df_angle_eval.query("key_num == 16").sort_values("n_fail", ascending=False)

+ 例えば、yaw_angle=2.897247が上手くいっていないので、その原因を確認

In [ ]:
np.where(np.abs(test_yaw_angles - 2.897247) < 0.001)

+ test_yaw_angles[165]の挙動を確認

In [ ]:
fix_test_yaw_angle = test_yaw_angles[166]
fix_param_pair = {"key_num": 16}

In [ ]:
# 1. 振りたいパラメータでCollisionDetectionを更新
target_conf = with_frozen_app_config(
    conf=app_config.CollisionDetection,
    **fix_param_pair
)
print(f"target_conf = {target_conf}")
app_config.CollisionDetection = target_conf

# 2. ループ内で何度も呼んでいるパラメータを変数に設定する
z_range=get_detectable_z_range(app_config.General, app_config.CollisionDetection)
max_dist=app_config.CollisionDetection.max_dist
grid_intervals=app_config.CollisionDetection.grid_intervals
min_radius=app_config.CollisionDetection.min_radius
max_radius=app_config.CollisionDetection.max_radius
vis_tree_depth = app_config.OctoTree.max_tree_depth - app_config.CollisionDetection.dialate_point_size
window = app_config.CollisionDetection.dialate_point_size
key_num = app_config.CollisionDetection.key_num


In [ ]:
# 3. そのパラメータで接触可能性探索の点群を生成するインスタンスを生成する
det_point_mobile_gen, det_point_immobile_gen = SubScrt.initialize_detectable_point_generators(
    machine_mobile_points=machine_mobile_points_measure,
    machine_immobile_points=machine_immobile_points_measure,
    detectable_tree_depth=vis_tree_depth,
    z_range=z_range,
    max_dist=max_dist,
    grid_intervals=grid_intervals,
    min_radius=min_radius,
    max_radius=max_radius,
    key_num=app_config.CollisionDetection.key_num,
    octotree_conf=app_config.OctoTree,
    dialate_point_size=app_config.CollisionDetection.dialate_point_size,
    offset_rotate_center=app_config.machine.offset_rotate_center,
)

In [ ]:
eval_points = create_eval_data_cylinder(
    machine_points=np.vstack([
        rotate_machine(machine_mobile_points_measure, -fix_test_yaw_angle, np.array([0.0, 0.0, 0.0])),
        machine_immobile_points_measure,
    ]),
    z_range=z_range,
    grid_intervals=grid_intervals,
    min_radius=min_radius,
    max_radius=max_radius,
    max_dist=max_dist,
)

In [ ]:
# B. その評価用の各LiDAR点群に対して接触可能性探索を行って、接触可能性有無で点群を分ける
oct_coords_fail, oct_coords_success = eval_detectable(
    eval_points=eval_points,
    yaw_angle=fix_test_yaw_angle,
    machine_immobile_points_measure=machine_immobile_points_measure,
    machine_mobile_points_measure=machine_mobile_points_measure,
    det_point_immobile_gen=det_point_immobile_gen,
    det_point_mobile_gen=det_point_mobile_gen,
    collision_detector=collision_detector,
    window=window,
    app_config=app_config,
)

In [ ]:
plot = k3d.plot()

plot += create_simple_k3d_points(machine_immobile_points_measure, color=0x00ff00, point_size=0.02)
plot += create_simple_k3d_points(machine_mobile_points_measure, color=0x00ff00, point_size=0.02)
plot += create_simple_k3d_points(oct_coords_fail, color=0xff0000, point_size=0.1)
plot += create_simple_k3d_points(oct_coords_success, color=0x0000ff, point_size=0.02)

plot.display()


In [ ]:
np.rad2deg(fix_test_yaw_angle)

+ 致命的な死角(ある方向が完全に検出できないような死角)ではなさそうであることは可視化で確認